In [1]:
import sys
!{sys.executable} -m pip install gradio python-dotenv openai

   ---------------------------------------- 0.0/20.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/20.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/20.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/20.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/20.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/20.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/20.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/20.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/20.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/20.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/20.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/20.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/20.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/20.1 MB ? eta -:--:--
   -----------------


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys
print("Python executable:", sys.executable)
print("Python path:", sys.path)

Python executable: c:\Users\User\AppData\Local\Programs\Python\Python314\python.exe
Python path: ['c:\\Users\\User\\AppData\\Local\\Programs\\Python\\Python314\\python314.zip', 'c:\\Users\\User\\AppData\\Local\\Programs\\Python\\Python314\\DLLs', 'c:\\Users\\User\\AppData\\Local\\Programs\\Python\\Python314\\Lib', 'c:\\Users\\User\\AppData\\Local\\Programs\\Python\\Python314', '', 'C:\\Users\\User\\AppData\\Roaming\\Python\\Python314\\site-packages', 'c:\\Users\\User\\AppData\\Local\\Programs\\Python\\Python314\\Lib\\site-packages']


In [3]:
import os
import glob
from dotenv import load_dotenv
from pathlib import Path
import gradio as gr
from openai import OpenAI

c:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Setting up

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY') or os.getenv('OPENAI API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

MODEL = "gpt-4.1-nano"
openai = OpenAI()

OpenAI API Key exists and begins sk-svcac


In [13]:
knowledge = {}
filenames = glob.glob("knowledge-base/employees/*") #get a list of string
for filename in filenames:
    name = Path(filename).stem.split(' ')[-1] #get the name of the file without extension and split by underscore
    # with open(filename, "r") as file:
    #     knowledge[name] = file.read() #store the content of the file in a dictionary with the name as the key
    with open(filename, "r", encoding="utf-8") as file: #whenever you have a text document, which format you want to read, you should specify the encoding. Otherwise, you may encounter errors when reading the file. UTF-8 is a common encoding that can handle a wide range of characters, including special characters and emojis.
        knowledge[name.lower()] = file.read() #store the content of the file in a dictionary with the name as the key
        

In [ ]:
knowledge["lancaster"]

In [16]:
filenames = glob.glob("knowledge-base/products/*")
for filename in filenames:
    name = Path(filename).stem.split(' ')[-1] #get the name of the file without extension and split by underscore
    with open(filename, "r", encoding="utf-8") as file:
        knowledge[name.lower()] = file.read() #store the content of the file in a dictionary with the name as the key

In [17]:
knowledge.keys()

dict_keys(['chen', 'harper', 'thomson', 'foster', 'lancaster', 'walker', 'rodriguez', 'park', 'kim', 'carter', 'tran', 'wilson', 'adams', 'liu', 'blake', 'bishop', 'zhang', 'anderson', 'johnson', 'thompson', "o'brien", 'rivera', 'patel', 'spencer', 'sharma', 'martinez', 'greene', 'trenton', 'williams', 'brooks', 'bizllm', 'carllm', 'claimllm', 'healthllm', 'homellm', 'lifellm', 'markellm', 'rellm'])

In [18]:
SYSTEM_PREFIX = """
You are Insurellm AI, an intelligent assistant for Insurellm, an Insurance Technology company.

Your role is to help employees by answering questions about:
- Insurellm products and services
- Company policies and procedures
- Employee-related information
- Insurance-related knowledge contained in the provided documents

Instructions:
1. Use the provided context as your primary source of information.
2. Answer clearly, accurately, and concisely.
3. If the answer is not available in the provided context, say:
   "I don't have enough information to answer that question."
4. Do not make up facts or assumptions.
5. When possible, base your response directly on the retrieved information.

Relevant Context:
"""


In [ ]:
def get_relevant_context_simple(message):
    text = ''.join(ch for ch in message if ch.isalpha() or ch.isspace()) #remove punctuation and special characters, the join function is used to concatenate the characters back into a string
    words = text.lower().split() #convert the text to lowercase and split it into words, this will help us to match the words with the keys in the knowledge dictionary
    relevant_context = [] #create an empty list to store the relevant context, return one item in the list, which is the content of the file that matches the word in the message
    for word in words:
        if word in knowledge:
            relevant_context.append(knowledge[word]) #check if the word is in the knowledge dictionary, if it is, append the corresponding value (the content of the file) to the relevant context list
    return relevant_context  

In [19]:
def get_relevant_context(message):
    text = ''.join(ch for ch in message if ch.isalpha() or ch.isspace())
    words = text.lower().split()
    return [knowledge[word] for word in words if word in knowledge]   


In [20]:
get_relevant_context("Who is lancaster?")

["# Avery Lancaster\n\n## Summary\n- **Date of Birth**: March 15, 1985\n- **Job Title**: Co-Founder & Chief Executive Officer (CEO)\n- **Location**: San Francisco, California\n- **Current Salary**: $225,000  \n\n## Insurellm Career Progression\n- **2015 - Present**: Co-Founder & CEO  \n  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  \n\n- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  \n  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the tech sector.  \n\n- **2010 - 2013**: Business Analyst at Edge Analytics  \n  Prior to joining Innovate, Avery worked as a Business Analyst, focusing on market 

In [21]:
get_relevant_context("Who is Lancaster and what is carllm?")

["# Avery Lancaster\n\n## Summary\n- **Date of Birth**: March 15, 1985\n- **Job Title**: Co-Founder & Chief Executive Officer (CEO)\n- **Location**: San Francisco, California\n- **Current Salary**: $225,000  \n\n## Insurellm Career Progression\n- **2015 - Present**: Co-Founder & CEO  \n  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  \n\n- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  \n  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the tech sector.  \n\n- **2010 - 2013**: Business Analyst at Edge Analytics  \n  Prior to joining Innovate, Avery worked as a Business Analyst, focusing on market 

In [22]:
def additional_context(message):
    relevant_context = get_relevant_context(message) #relevant_context is a list of strings, each string is the content of a file that matches a word in the message, if no word in the message matches a key in the knowledge dictionary, relevant_context will be an empty list
    if not relevant_context:
        result = "There is no additional context relevant to the user's question."
    else:
        result = "The following additional context might be relevant in answering the user's question:\n\n"
        result += "\n\n".join(relevant_context)
    return result

In [23]:
print(additional_context("Who is Alex Lancaster?"))

The following additional context might be relevant in answering the user's question:

# Avery Lancaster

## Summary
- **Date of Birth**: March 15, 1985
- **Job Title**: Co-Founder & Chief Executive Officer (CEO)
- **Location**: San Francisco, California
- **Current Salary**: $225,000  

## Insurellm Career Progression
- **2015 - Present**: Co-Founder & CEO  
  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  

- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  
  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the tech sector.  

- **2010 - 2013**: Business Analyst at Edge Analytics  
  Prior to joini

In [24]:
def chat(message, history):
    system_message = SYSTEM_PREFIX + additional_context(message) #the system message is the combination of the system prefix and the additional context, which is the relevant information retrieved from the knowledge base based on the user's question
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}] #the messages is a list of dictionaries, each dictionary represents a message in the conversation, the system message is the first message, followed by the conversation history, and the user's message is the last message
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content


In [26]:
view = gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
